# SnapCard: Fine-tuning mT5-base with LoRA

This notebook trains two LoRA adapters on top of `google/mt5-base`:
1. **Title adapter** — generates a Russian product title from `caption_ru + category`.
2. **Description adapter** — generates a Russian product description from `caption_ru + category + title`.

Run in Google Colab with a GPU runtime (T4 is sufficient).

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate sacrebleu rouge-score
# Colab ships an old torchao that conflicts with recent peft; we don't use it
!pip uninstall -y -q torchao

## 2. Mount Google Drive

Upload the four JSONL files generated by `training/prepare_mt5_dataset.py` to your Drive, then update `DATA_DIR`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/snapcard_mt5_data'  # change if needed
OUTPUT_DIR = '/content/drive/MyDrive/snapcard_mt5_adapters'

## 3. Imports and hyperparameters

In [ ]:
import json
import random
import time
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

BASE_MODEL = "google/mt5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Hyperparameters
EPOCHS = 20
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
MAX_SOURCE_LENGTH = 128
MAX_TITLE_LENGTH = 64
MAX_DESCRIPTION_LENGTH = 200
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q", "v"]

print(f"Device: {DEVICE}")


## 4. Load datasets

In [ ]:
title_dataset = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/mt5_title_train.jsonl",
    "val": f"{DATA_DIR}/mt5_title_val.jsonl",
})

desc_dataset = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/mt5_description_train.jsonl",
    "val": f"{DATA_DIR}/mt5_description_val.jsonl",
})

print(title_dataset)
print(desc_dataset)


## 5. Tokenizer and preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess_title(examples):
    inputs = examples["input"]
    targets = examples["target"]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=MAX_TITLE_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def preprocess_description(examples):
    inputs = examples["input"]
    targets = examples["target"]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(targets, max_length=MAX_DESCRIPTION_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

title_tokenized = title_dataset.map(preprocess_title, batched=True, remove_columns=title_dataset["train"].column_names)
desc_tokenized = desc_dataset.map(preprocess_description, batched=True, remove_columns=desc_dataset["train"].column_names)


## 6. LoRA configuration

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(DEVICE)


## 7. Train title adapter

In [ ]:
title_model = get_peft_model(base_model, lora_config)

training_args = TrainingArguments(
    output_dir="/content/mt5_title_checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

title_trainer = Trainer(
    model=title_model,
    args=training_args,
    train_dataset=title_tokenized["train"],
    eval_dataset=title_tokenized["val"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=title_model),
)

title_trainer.train()

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
title_model.save_pretrained(f"{OUTPUT_DIR}/snapcard_mt5_title_lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/snapcard_mt5_title_lora")
print(f"Title adapter saved to {OUTPUT_DIR}/snapcard_mt5_title_lora")


## 8. Train description adapter

Reload the base model to avoid carrying over title adapter weights.

In [ ]:
del title_model, title_trainer
import gc
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(DEVICE)
desc_model = get_peft_model(base_model, lora_config)

training_args = TrainingArguments(
    output_dir="/content/mt5_description_checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

desc_trainer = Trainer(
    model=desc_model,
    args=training_args,
    train_dataset=desc_tokenized["train"],
    eval_dataset=desc_tokenized["val"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=desc_model),
)

desc_trainer.train()

desc_model.save_pretrained(f"{OUTPUT_DIR}/snapcard_mt5_description_lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/snapcard_mt5_description_lora")
print(f"Description adapter saved to {OUTPUT_DIR}/snapcard_mt5_description_lora")


## 9. Sample inference

In [ ]:
from peft import PeftModel

def generate(model, prompt, max_length=128):
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(DEVICE)
    outputs = model.generate(**inputs, max_new_tokens=max_length, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# Load title adapter
base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(DEVICE)
title_model = PeftModel.from_pretrained(base, f"{OUTPUT_DIR}/snapcard_mt5_title_lora").to(DEVICE)
title_model.eval()

# Load description adapter on a fresh base
base2 = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(DEVICE)
desc_model = PeftModel.from_pretrained(base2, f"{OUTPUT_DIR}/snapcard_mt5_description_lora").to(DEVICE)
desc_model.eval()

samples = desc_dataset["val"].select(range(5))
for rec in samples:
    title = generate(title_model, rec["input"].replace("Описание товара", "Заголовок товара").split(". Заголовок")[0] + ".", max_length=MAX_TITLE_LENGTH)
    desc_prompt = rec["input"].replace("{title}", title)
    desc = generate(desc_model, desc_prompt, max_length=MAX_DESCRIPTION_LENGTH)
    print(f"Caption: {rec['input']}")
    print(f"Generated title: {title}")
    print(f"Generated desc:  {desc}")
    print(f"Reference title: {rec['target'] if 'target' in rec else 'N/A'}")
    print("-" * 80)


## 10. Evaluate on validation set

Compute BLEU-4 and ROUGE-L.

In [ ]:
def evaluate(model, dataset, max_length=128):
    predictions, references = [], []
    times = []
    for rec in dataset:
        start = time.time()
        pred = generate(model, rec["input"], max_length=max_length)
        times.append(time.time() - start)
        predictions.append(pred)
        references.append(rec["target"])

    bleu = corpus_bleu(predictions, [references])
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    rouge_scores = [scorer.score(ref, pred)["rougeL"].fmeasure for ref, pred in zip(references, predictions)]
    avg_rouge = sum(rouge_scores) / len(rouge_scores)

    return {
        "bleu4": bleu.score / 100,
        "rouge_l": avg_rouge,
        "avg_latency_ms": sum(times) / len(times) * 1000,
    }

print("Title metrics:", evaluate(title_model, title_dataset["val"], max_length=MAX_TITLE_LENGTH))
print("Description metrics:", evaluate(desc_model, desc_dataset["val"], max_length=MAX_DESCRIPTION_LENGTH))


## 11. Download adapters

The adapters are saved in your Google Drive at `{OUTPUT_DIR}`. Download the two folders (`snapcard_mt5_title_lora` and `snapcard_mt5_description_lora`) and place them in the project under:

```
backend/model_cache/
├── snapcard_mt5_title_lora/
└── snapcard_mt5_description_lora/
```

Then set environment variables or use default paths in `backend/app/config.py`.